CONOR MURRAY - Convolutional Neural Networks

In [41]:
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

In [49]:
df = pd.read_csv("pokemon_all.csv")
mask = df["Type_2"].notna()
df = df[mask]
df.head()

,Number,Name,Type_1,Type_2,Total,HP,Attack,Defense,Sp_Atk,Sp_Def,...,Color,hasGender,Pr_Male,Egg_Group_1,Egg_Group_2,hasMegaEvolution,Height_m,Weight_kg,Catch_Rate,Body_Style
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,...,Green,True,0.875,Monster,Grass,False,0.71,6.9,45,quadruped
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,...,Green,True,0.875,Monster,Grass,False,0.99,13.0,45,quadruped
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,...,Green,True,0.875,Monster,Grass,True,2.01,100.0,45,quadruped
5,6,Charizard,Fire,Flying,534,78,84,78,109,85,...,Red,True,0.875,Monster,Dragon,True,1.70,90.5,45,bipedal_tailed
11,12,Butterfree,Bug,Flying,395,60,45,50,90,80,...,White,True,0.500,Bug,NaN,False,1.09,32.0,45,four_wings


In [43]:
image_folder = "pokemon_png"
numbers = df["Number"].unique()
image_list = []
loaded_numbers = []
for num in numbers:
    file_path = image_folder + "/" + str(int(num)) + ".png"
    try:
        img = Image.open(file_path)      
        img = img.convert("L")           
        img_array = np.array(img)        
        image_list.append(img_array) 
        loaded_numbers.append(num)
    except FileNotFoundError:
        pass
images_3d = np.array(image_list)
max_value = images_3d.max()
min_value = images_3d.min()
images_3d = images_3d / (max_value - min_value)
print('value at [0, 70, 35] is: ', images_3d[0, 70, 35])

value at [0, 70, 35] is:  0.7019607843137254


In [44]:
mask = df["Number"].isin(loaded_numbers)
y_df = pd.get_dummies(df[mask]["Type_2"])
y = y_df.to_numpy()
print("Categories:", list(y_df.columns))
print("Shape of y:", y.shape)

Categories: ['Bug', 'Dark', 'Dragon', 'Electric', 'Fairy', 'Fighting', 'Fire', 'Flying', 'Ghost', 'Grass', 'Ground', 'Ice', 'Normal', 'Poison', 'Psychic', 'Rock', 'Steel', 'Water']
Shape of y: (342, 18)


In [45]:
num_classes = y.shape[1]
model = models.Sequential()
model.add(layers.Conv2D(16, (3, 3), activation="relu", input_shape=(256,256,1)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(32, (3, 3), activation="relu"))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation="relu"))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Flatten())
model.add(layers.Dense(64, activation="relu"))
model.add(layers.Dense(num_classes, activation="softmax"))
model.summary()

C:\Users\conor\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)                   │ (None, 254, 254, 16)        │             160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_12 (MaxPooling2D)      │ (None, 127, 127, 16)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_13 (Conv2D)                   │ (None, 125, 125, 32)        │           4,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_13 (MaxPooling2D)      │ (None, 62, 62, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_14 (Conv2D)                   │ (None, 60, 60, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_14 (MaxPooling2D)      │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_4 (Flatten)                  │ (None, 57600)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 64)                  │       3,686,464 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (None, 18)                  │           1,170 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,710,930 (14.16 MB)

 Trainable params: 3,710,930 (14.16 MB)

 Non-trainable params: 0 (0.00 B)

In [46]:
images_3d = images_3d.reshape(images_3d.shape[0], 256, 256, 1)
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])
history = model.fit(images_3d, y, epochs=10)

Epoch 1/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 171ms/step - accuracy: 0.2018 - loss: 2.7788
Epoch 2/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 128ms/step - accuracy: 0.2485 - loss: 2.4848
Epoch 3/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 129ms/step - accuracy: 0.3099 - loss: 2.1777
Epoch 4/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 129ms/step - accuracy: 0.5409 - loss: 1.6476
Epoch 5/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 130ms/step - accuracy: 0.7895 - loss: 0.8153
Epoch 6/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 129ms/step - accuracy: 0.9415 - loss: 0.2500
Epoch 7/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 130ms/step - accuracy: 0.9942 - loss: 0.0383
Epoch 8/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 129ms/step - accuracy: 1.0000 - loss: 0.0040
Epoch 9/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 129ms/step - accuracy: 1.0000 - loss: 8.8155e-04
Epoch 10/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 129ms/step - accuracy: 1.0000 - loss: 5.0020e-04


In [47]:
predict_img = Image.open("pokemon_predict_image.png")
predict_img = predict_img.convert("L")
predict_array = np.array(predict_img)
max_value = predict_array.max()
min_value = predict_array.min()
predict_array = predict_array / (max_value - min_value)
predict_array = predict_array.reshape(1, 256, 256, 1)
predictions = model.predict(predict_array)
print('Prediction probabilities: ', predictions)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
Prediction probabilities:  [[1.0126548e-14 5.7960749e-15 1.1055556e-06 3.9108993e-15 6.4667507e-12
  5.7972366e-11 5.3760535e-16 9.9999785e-01 1.0326270e-06 3.0886987e-08
  4.6108565e-11 8.1112121e-09 1.0438903e-17 1.1907318e-11 1.7558269e-16
  5.2718335e-15 3.3478209e-10 3.9431579e-12]]


In [48]:
best_index = np.argmax(predictions)
class_names = list(y_df.columns)
predicted_class = class_names[best_index]
predicted_prob = predictions [0, best_index]
print('Most likely class: ', predicted_class)
print('Probability: ', predicted_prob)

Most likely class:  Flying
Probability:  0.99999785
